In [ ]:
import random
import time
from gpiozero import Button, RotaryEncoder
PIN_CLK=22
PIN_DT=27
PIN_SW=17

def generar_instrucciones(cant_pasos=3):
    acciones_posibles = ["boton","girar"]
    direcciones_posibles = ["derecha","izquierda"]
    
    lista_instrucciones = []
    for i in range(cant_pasos):
        tipo = random.choice(acciones_posibles)
        if tipo == "boton":
            repeticiones = random.randint(1, 5)
            lista_instrucciones.append(
                {"accion":"boton", "direccion":None, "repeticiones": repeticiones}
                )
        else:
            direccion = random.choice(direcciones_posibles)
            repeticiones = random.randint(1,3)
            lista_instrucciones.append(
                {
                    "accion":"girar",
                    "direccion": direccion,
                    "repeticiones": repeticiones,
                    }
                )
    return lista_instrucciones
instrucciones = generar_instrucciones(cant_pasos=3)

        
def ejecutar_minijuego_simon(max_time=15):
    rotor=RotaryEncoder(22,27,max_steps=0)
    boton=Button(17,pull_up=True, bounce_time=0.1)
    instrucciones = generar_instrucciones(cant_pasos=3)
    pasos_totales = len(instrucciones)
    pasos_completados = 0
    print(f"inicio simon dice")
    start_time = time.time()
    
    try:
        for idx, inst in enumerate(instrucciones, start=1):
            if (time.time() - start_time) >= max_time:
                break
            print(f"pasos totales")
            if inst ["accion"] == "boton":
                print(f"PRESIONA el boton{inst['repeticiones']} vez/veces")
            else:
                print(f"Girar a la {inst['direccion']} {inst['repeticiones']} vez/veces")
            rotor.steps=0
            ultimo_paso_rotor = 0
            pulsaciones_actuales = 0
            instruccion_cumplida = False
            
            while not instruccion_cumplida:
                if(time.time() - start_time) >= max_time:
                    break
                if inst["accion"] == "girar":
                    delta = rotor.steps - ultimo_paso_rotor
                    
                    if delta != 0:
                        print(delta)
                        if inst["direccion"] == "derecha":
                            if delta > 0:
                                if rotor.steps >= inst["repeticiones"]:
                                    instruccion_cumplida = True
                            else:
                                print("Error! Giraste a la izquierda")
                                return {"Result": "Lose", "Score":0}
                        
                        elif inst["direccion"] == "izquierda":
                            if delta < 0:
                                if abs(rotor.steps) >= inst["repeticiones"]:
                                    instruccion_cumplida = True
                            else:
                                print("Error! Giraste a la derecha")
                                return {"Result": "Lose", "Score":0}
                        ultimo_paso_rotor = rotor.steps
                    
                    if boton.is_pressed:
                        print("Error! presionaste el boton en lugar de girar")
                        return {"Result": "Lose", "Score":0}
                elif inst["accion"] == "boton":
                    if boton.is_pressed:
                        pulsaciones_actuales += 1
                        print(f" Pulsacion {pulsaciones_actuales}/{inst["repeticiones"]}")
                        
                        if pulsaciones_actuales >= inst["repeticiones"]:
                            instruccion_cumplida = True
                        time.sleep(0.25)
                    if rotor.steps !=0:
                        print("Error! Giraste la perilla en lugar de presionar")
                        return {"Result": "Lose", "Score":0}
                time.sleep(0.01)
            if instruccion_cumplida:
                pasos_completados += 1
                print("Paso correcto!")
            else:
                break
    finally:
        rotor.close()
        boton.close()
    if pasos_completados == pasos_totales:
        print("\n Mini juego completado con exito!")
        return {"Result": "Win", "Score":100}
    else:
        score_parcial = int((pasos_completados / pasos_totales) * 99)
        score_final = max(1, score_parcial)
        print(f"\n tiempo agotado! completaste {pasos_completados}/{pasos_totales}")
        return {"Result": "TimeOut", "Score": score_final}
if __name__ == "__main__":
    res = ejecutar_minijuego_simon(max_time=15)
    print("\n Respuesta a enviar al servidor:")
    print(res)
                        
                            
                    
                    
